In [1]:
from sklearn import datasets
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import math

In [2]:
mean = [0 for _ in range(100)]


In [3]:
cov_1 = [0.5 for _ in range(100)]
cov_mat_1 = np.diag(cov_1)
cov_mat_1

array([[0.5, 0. , 0. , ..., 0. , 0. , 0. ],
       [0. , 0.5, 0. , ..., 0. , 0. , 0. ],
       [0. , 0. , 0.5, ..., 0. , 0. , 0. ],
       ...,
       [0. , 0. , 0. , ..., 0.5, 0. , 0. ],
       [0. , 0. , 0. , ..., 0. , 0.5, 0. ],
       [0. , 0. , 0. , ..., 0. , 0. , 0.5]])

In [4]:
cov_2 = [1.0 for _ in range(100)]
cov_mat_2 = np.diag(cov_2)
cov_mat_2

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]])

In [5]:
def generate_data():
    A1 = np.random.multivariate_normal(mean, cov_mat_1 ,20)  #normal
    A2 = np.random.multivariate_normal(mean, cov_mat_2 ,20) 
    l_1 = [f'f{i}' for i in range(1, 101)]
    df_A1 = pd.DataFrame(A1, columns = l_1)
    df_A2 = pd.DataFrame(A2, columns = l_1) 
    df = pd.concat([df_A1, df_A2], axis = 0, ignore_index = True)
    X = df.to_numpy(dtype = None, copy = False)
    return X

In [6]:
X = generate_data()
X

array([[-0.37252651, -0.70939657, -0.01764804, ..., -0.19193435,
        -0.52235992,  0.32113022],
       [ 0.79014995,  1.4777664 , -0.41364982, ...,  0.59323586,
         0.12861753, -0.11776202],
       [-0.3224147 ,  0.56461283, -0.94673475, ..., -0.76954556,
        -0.68435185, -0.37691644],
       ...,
       [ 0.34711133,  0.16967467,  0.573534  , ...,  1.41341613,
         0.67717908,  0.760256  ],
       [-0.70371526, -1.42190571,  0.1477795 , ...,  0.18358112,
        -0.00201661,  0.28219226],
       [ 0.6940685 ,  1.74729729, -0.26656846, ...,  0.85406851,
         1.37532448,  0.64038397]])

In [7]:
def calculate_dist_mat(X):
    dist_mat= np.zeros((len(X),len(X)))
    for i in range(1, len(dist_mat[0])):
        for j in range(i):
            for k in range(len(dist_mat[0])):
                if k == i or k == j:
                    continue
                dist_mat[i][j] += abs(np.linalg.norm(X[i] - X[k]) - np.linalg.norm(X[j] - X[k]))
            dist_mat[i][j] /= (len(dist_mat[0])-2)
    return dist_mat

In [8]:
def ret_min(a, b): # Returns minimum of a and b
  if a>b:
    return b
  return a

In [9]:
from collections.abc import Iterable

def flatten(lis):
     for item in lis:
         if isinstance(item, Iterable) and not isinstance(item, str):
             for x in flatten(item):
                 yield x
         else:        
             yield item

In [10]:
def calculate_Madd(X,i,j):
    madd = 0.0
    for k in range(len(X)):
        if k == i or k == j:
            continue
        madd += abs(np.linalg.norm(X[i] - X[k]) - np.linalg.norm(X[j] - X[k]))
    madd /= (len(X) - 2)
    return madd

In [12]:
def within_cluster_distance(X,hierarchy):
    w_max = 0
    for i in range(len(hierarchy)):
    s1 = 0
    cluster = sorted(list(flatten(hierarchy[i])))
    
    if len(cluster) == 1:
            continue
    else:
      for j in range(len(cluster)-1):
        for k in range(j+1,len(cluster)):
           s1 += calculate_Madd(X,cluster[j],cluster[k])
      s1 /=  (len(cluster) * (len(cluster)-1))/2 
    if s1>w_max:
      w_max = s1
  return w_max

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 16)

In [ ]:
def between_cluster_distance(X,hierarchy):
  b_min = float("inf")
  for i in range(len(hierarchy)-1):
    cluster_1 = sorted(list(flatten(hierarchy[i])))
    for j in range(i+1,len(hierarchy)):
      s2 = 0.0
      cluster_2 = sorted(list(flatten(hierarchy[j])))
      for k in range(len(cluster_1)):
        for l in range(len(cluster_2)):
          s2 += calculate_Madd(X,cluster_1[k], cluster_2[l])
      s2 /= (len(cluster_1) * len(cluster_2))
      if s2<b_min:
        b_min = s2
           
  return b_min

In [ ]:
def clustering(X,matrix, hierarchy,d,k_max):
  
  # Base case
  if len(matrix) == 2:
    min_i = 0
    min_j = 1
    hierarchy = hierarchy[:min_i] + [hierarchy[min_i], hierarchy[min_j]] + hierarchy[min_j+1:]
    B = d[-1]
    d = d[:-1]
    w_max = within_cluster_distance(X,hierarchy)
    d += [B/w_max]

    max = d[0]
    index = 0
    for i in range(1,len(d)):
        if d[i] >= max:
            max = d[i]
            index = i
    return k_max-index
    


  min = matrix[1][0] #initializing minimum distance
  min_i, min_j = 1, 0

  for i in range(1, len(matrix[0])):
    for j in range(i):
      if i>j and ((i-j) == 1):
        if matrix[i][j]<min:
          min = matrix[i][j]
          min_i = i
          min_j = j
 
  

  # Swapping min_i, min_j in order to maintain min_i < min_j
  if min_i > min_j:
    a = min_i
    min_i = min_j
    min_j = a
  
  
  hierarchy = hierarchy[:min_i] + [[hierarchy[min_i], hierarchy[min_j]]] + hierarchy[min_j+1:]

  dist_mat = np.copy(matrix) # Copying the original matrix so that original matrix is not affected by further operations
  dist_mat = np.delete(dist_mat, (min_j), axis=0) # Deleting  min_j row
  dist_mat = np.delete(dist_mat, (min_j), axis=1) # Deleting  min_j column

  """
  # Copying the remaining matrix to new distance matrix for recursion
  dist_mat_2 = np.zeros((len(matrix)-1, len(matrix)-1))
  for i in range(len(matrix)-2):
    for j in range(len(matrix)-2):
      dist_mat_2[i][j] = dist_mat[i][j]
  """

  # Deleting dist_mat in order to save space
  #dist_mat = np.delete(dist_mat, [i for i in range(len(dist_mat))], 0)

  dist_min_i = [] # List of distances from point min_i to all other points except min_j
  dist_min_j = [] # List of distances from point min_j to all other points except min_i

  for i in range(len(matrix)):
    if i<min_i:
      dist_min_i += [matrix[min_i][i]]
      dist_min_j += [matrix[min_j][i]]
    elif i == min_i or i == min_j:
      continue
    elif i<min_j:
      dist_min_i += [matrix[i][min_i]]
      dist_min_j += [matrix[min_j][i]]
    else:
      dist_min_i += [matrix[i][min_i]]
      dist_min_j += [matrix[i][min_j]]
  
  # Filling the last row of new distance matrix
  for i in range(len(dist_min_i)):
    if i < min_i:
      dist_mat[min_i][i] = ret_min(dist_min_i[i], dist_min_j[i])
    else:
      dist_mat[i+1][min_i] = ret_min(dist_min_i[i], dist_min_j[i])
  
  dist_min_i = []
  dist_min_j = []
  
  
  l_h = len(hierarchy)
  if (l_h <= k_max and l_h > 1):
    b_min = between_cluster_distance(X,hierarchy)
    w_max = within_cluster_distance(X,hierarchy)
    d += [b_min/w_max]
    if(l_h == 2):
        d += [b_min]
 

    
    
    
  
    
  
 
  opt_max = clustering(X,dist_mat, hierarchy,d,k_max) # The recusion call

  return opt_max

In [ ]:
opt_cluster = []
#hier1 = [[i] for i in range(len(dist_mat_1))]
for _ in range(50):
    X = generate_data()
    dist_mat = calculate_dist_mat(X)
    hier1 = [[i] for i in range(len(dist_mat))]
    opt_cluster += [ clustering(X,dist_mat, hier1,[],20)]

In [ ]:
opt_cluster

In [ ]:
# Creating histogram
n_bins = 20
fig, axs = plt.subplots(1, 1,
                        figsize =(10, 7),
                        tight_layout = True)
 
axs.hist(opt_cluster, bins = n_bins)
 
# Show plot
plt.show()